# Lab #4 (v2): Instacart 장바구니 데이터를 이용한 연관규칙 분석 (정답)

실제 온라인 식료품 마켓인 **Instacart**의 거래 데이터를 **Kaggle API**를 통해 직접 다운로드하여 분석해 보겠습니다. 이 데이터셋은 여러 파일로 나뉘어 있어, 데이터를 병합하고 가공하는 과정이 추가됩니다.

### 과제 목표
1. Kaggle API를 설정하고 Instacart 데이터셋을 다운로드합니다.
2. 여러 CSV 파일(`orders`, `products`, `order_products__prior`)을 로드하고, `merge`를 통해 분석에 필요한 데이터프레임을 생성합니다.
3. `Apriori` 알고리즘을 사용하여 빈번하게 발생하는 상품 조합을 찾습니다.
4. `association_rules`를 통해 연관규칙을 생성하고, **신뢰도(Confidence)**와 **향상도(Lift)**를 기준으로 의미 있는 규칙을 필터링합니다.
5. 특정 상품(예: 'Banana')과 연관성이 높은 상품들을 찾아내고, 시각화를 통해 결과를 분석합니다.

### [준비] Kaggle API 설정

In [2]:
import kaggle
import os
import json

In [ ]:
# kaggle.json 파일 경로
kaggle_json_path = os.path.expanduser('~/.kaggle/kaggle.json')
# kaggle.json 파일이 없으면 생성
if not os.path.exists(kaggle_json_path):
    os.makedirs(os.path.dirname(kaggle_json_path), exist_ok=True)
    
    # Kaggle API 인증 정보 입력 받기
    username = input("Kaggle 사용자 이름을 입력하세요: ")
    key = input("Kaggle API 키를 입력하세요: ")
    
    # kaggle.json 파일 생성
    kaggle_json = {
        "username": username,
        "key": key
    }
    
    with open(kaggle_json_path, 'w') as f:
        json.dump(kaggle_json, f)
    
    # 파일 권한 설정 (Kaggle API 요구사항)
    os.chmod(kaggle_json_path, 0o600)
    
    print("Kaggle API 인증이 완료되었습니다.")
else:
    print("이미 Kaggle API 인증이 설정되어 있습니다.")


### [준비] 데이터 다운로드 및 로드

In [3]:
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import plotly.express as px

# Kaggle 데이터셋 다운로드 및 압축 해제
#(도커/로컬 환경에 따라 저장위치를 바꾸세요)
base_path = "../datasets/ml/instacart/"

In [ ]:
# ggle API를 사용하여 데이터셋 다운로드 
kaggle.api.dataset_download_files('yasserh/instacart-online-grocery-basket-analysis-dataset', path=base_path, unzip=True)

In [5]:
# 데이터 로드
orders_df = pd.read_csv(os.path.join(base_path,'orders.csv'))
products_df = pd.read_csv(os.path.join(base_path,'products.csv'))
order_products_prior_df = pd.read_csv(os.path.join(base_path,'order_products__prior.csv'))

print('--- orders_df ---')
print(orders_df.head())
print('\n--- products_df ---')
print(products_df.head())
print('\n--- order_products_prior_df ---')
print(order_products_prior_df.head())

--- orders_df ---
   order_id  user_id eval_set  order_number  order_dow  order_hour_of_day  \
0   2539329        1    prior             1          2                  8   
1   2398795        1    prior             2          3                  7   
2    473747        1    prior             3          3                 12   
3   2254736        1    prior             4          4                  7   
4    431534        1    prior             5          4                 15   

   days_since_prior_order  
0                     NaN  
1                    15.0  
2                    21.0  
3                    29.0  
4                    28.0  

--- products_df ---
   product_id                                       product_name  aisle_id  \
0           1                         Chocolate Sandwich Cookies        61   
1           2                                   All-Seasons Salt       104   
2           3               Robust Golden Unsweetened Oolong Tea        94   
3           4  Sma

### [문제 1] 데이터 병합 및 샘플링

In [6]:
# 1. 20,000개 주문 샘플링
prior_orders_df = orders_df[orders_df['eval_set'] == 'prior']
sampled_orders_df = prior_orders_df.sample(n=20000, random_state=42)

# 2. 주문-상품 데이터 병합
df_order_products = pd.merge(sampled_orders_df, order_products_prior_df, on='order_id', how='inner')

# 3. 상품 이름 데이터 병합
df_merged = pd.merge(df_order_products, products_df, on='product_id', how='inner')

# 최종 데이터 확인
print(df_merged.shape)
df_merged[['order_id', 'product_id', 'product_name']].head()

(200704, 13)


,order_id,product_id,product_name
0,3278314,43789,Organic Basil
1,3278314,21137,Organic Strawberries
2,3278314,30391,Organic Cucumber
3,3278314,18883,Honeydew Chunks
4,3278314,46979,Asparagus


### [문제 2] 트랜잭션 데이터로 변환

In [7]:
# order_id를 기준으로 product_name을 리스트로 묶기
transactions = df_merged.groupby('order_id')['product_name'].apply(list).tolist()

# TransactionEncoder를 사용하여 One-Hot 인코딩
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

df_encoded.head()

,#2 Coffee Filters,#4 Natural Brown Coffee Filters,& Go! Hazelnut Spread + Pretzel Sticks,0 Calorie Acai Raspberry Water Beverage,0 Calorie Fuji Apple Pear Water Beverage,0 Calorie Strawberry Dragonfruit Water Beverage,0% Fat Black Cherry Greek Yogurt y,0% Fat Blueberry Greek Yogurt,0% Fat Free Organic Milk,0% Fat Greek Yogurt Black Cherry on the Bottom,...,w/Banana Pulp Free Juice,with Crispy Almonds Cereal,with Dawn Action Pacs Fresh Scent Dishwasher Detergent Pacs,with Olive Oil Mayonnaise,with Olive Oil Mayonnaise Dressing,with Seasoned Roasted Potatoes Scrambled Eggs & Sausage,with Sweet & Smoky BBQ Sauce Cheeseburger Sliders,with Xylitol Cinnamon 18 Sticks Sugar Free Gum,with Xylitol Original Flavor 18 Sticks Sugar Free Gum,with Xylitol Unwrapped Spearmint 50 Sticks Sugar Free Gum
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


### [문제 3] Apriori 알고리즘 적용 및 연관규칙 생성

In [8]:
# Apriori 알고리즘 적용
frequent_itemsets = apriori(df_encoded, min_support=0.01, use_colnames=True)

# 연관규칙 생성
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.5)

# 신뢰도(confidence)와 향상도(lift) 기준으로 정렬하여 상위 10개 규칙 확인
rules.sort_values(by=['lift', 'confidence'], ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
24,(Organic Raspberries),(Organic Strawberries),0.04090,0.08370,0.01045,0.255501,3.052583,1.0,0.007027,1.230761,0.701083,0.091546,0.187495,0.190176
25,(Organic Strawberries),(Organic Raspberries),0.08370,0.04090,0.01045,0.124851,3.052583,1.0,0.007027,1.095927,0.733830,0.091546,0.087531,0.190176
5,(Organic Raspberries),(Bag of Organic Bananas),0.04090,0.11550,0.01355,0.331296,2.868362,1.0,0.008826,1.322707,0.679146,0.094855,0.243975,0.224306
4,(Bag of Organic Bananas),(Organic Raspberries),0.11550,0.04090,0.01355,0.117316,2.868362,1.0,0.008826,1.086572,0.736426,0.094855,0.079675,0.224306
14,(Organic Fuji Apple),(Banana),0.02705,0.14375,0.01005,0.371534,2.584586,1.0,0.006162,1.362445,0.630136,0.062519,0.266025,0.220724
15,(Banana),(Organic Fuji Apple),0.14375,0.02705,0.01005,0.069913,2.584586,1.0,0.006162,1.046085,0.716018,0.062519,0.044055,0.220724
2,(Organic Hass Avocado),(Bag of Organic Bananas),0.06840,0.11550,0.02010,0.293860,2.544239,1.0,0.012200,1.252584,0.651519,0.122711,0.201650,0.233943
3,(Bag of Organic Bananas),(Organic Hass Avocado),0.11550,0.06840,0.02010,0.174026,2.544239,1.0,0.012200,1.127881,0.686213,0.122711,0.113381,0.233943
22,(Organic Hass Avocado),(Organic Strawberries),0.06840,0.08370,0.01350,0.197368,2.358046,1.0,0.007775,1.141620,0.618205,0.097403,0.124052,0.179329
23,(Organic Strawberries),(Organic Hass Avocado),0.08370,0.06840,0.01350,0.161290,2.358046,1.0,0.007775,1.110754,0.628528,0.097403,0.099711,0.179329


### [문제 4] 특정 상품 관련 규칙 분석

In [9]:
# 조건절에 'Banana'가 포함된 규칙 찾기
is_banana_antecedent = rules['antecedents'].apply(lambda x: 'Banana' in x)
banana_rules = rules[is_banana_antecedent]

print("'Banana'와 함께 구매되는 상품 Top 10 (by Lift)")
banana_rules.sort_values(by='lift', ascending=False).head(10)

'Banana'와 함께 구매되는 상품 Top 10 (by Lift)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
15,(Banana),(Organic Fuji Apple),0.14375,0.02705,0.01005,0.069913,2.584586,1.0,0.006162,1.046085,0.716018,0.062519,0.044055,0.220724
12,(Banana),(Organic Avocado),0.14375,0.05435,0.01710,0.118957,2.188712,1.0,0.009287,1.073330,0.634289,0.094475,0.068320,0.216792
16,(Banana),(Strawberries),0.14375,0.04215,0.01205,0.083826,1.988757,1.0,0.005991,1.045489,0.580640,0.069313,0.043510,0.184855
8,(Banana),(Large Lemon),0.14375,0.04890,0.01310,0.091130,1.863608,1.0,0.006071,1.046465,0.541205,0.072960,0.044402,0.179512
11,(Banana),(Limes),0.14375,0.04295,0.01100,0.076522,1.781647,1.0,0.004826,1.036354,0.512376,0.062607,0.035078,0.166317


### [문제 5] 분석 결과 시각화

In [10]:
# 시각화를 위해 frozenset을 문자열로 변환
rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

# Scatter plot 생성
fig = px.scatter(rules, 
                 x="support", 
                 y="confidence", 
                 color="lift", 
                 hover_data=['antecedents_str', 'consequents_str'],
                 title="Instacart Association Rules (Sampled)")
fig.show()